In [12]:
import zipfile
import os
import numpy as np
from pathlib import Path
from tqdm import tqdm

import yaml

import sys
PROJECT_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()


# Agregar el directorio raíz del proyecto al sys.path para importar módulos personalizados
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.utils import process_and_store_fracatlas, collect_tasks

In [18]:
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
# Cargar la configuración del proyecto desde el archivo YAML
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

In [28]:
RAW_PATH = Path(PROJECT_ROOT) / config['paths']["raw_dataset_path"]
PROCESSED_PATH = Path(PROJECT_ROOT) / config['paths']["processed_dataset_path"]
TARGET_DIR_FRACTURED = PROCESSED_PATH / 'fractured'
TARGET_DIR_NON_FRACTURED = PROCESSED_PATH / 'non_fractured'

os.makedirs(RAW_PATH, exist_ok=True)
os.makedirs(PROCESSED_PATH, exist_ok=True)
os.makedirs(TARGET_DIR_FRACTURED, exist_ok=True)
os.makedirs(TARGET_DIR_NON_FRACTURED, exist_ok=True)

## Descarga y preprocesamiento de dataset FractAtlas

In [29]:
#!/bin/bash
!curl -L -o {RAW_PATH}/fractatlas.zip\
  https://www.kaggle.com/api/v1/datasets/download/mahmudulhasantasin/fracatlas-original-dataset

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed
  0      0   0      0   0      0      0      0                              0
100 322.7M 100 322.7M   0      0 43.98M      0   00:07   00:07         55.79M


In [ ]:
# Configuración de rutas
ZIP_ATLAS = RAW_PATH / 'fractatlas.zip'
RAW_DIR_ATLAS = RAW_PATH / 'FractAtlas_raw'

# 1. Extraer el nuevo dataset
os.makedirs(RAW_DIR_ATLAS, exist_ok=True)
if os.path.exists(ZIP_ATLAS):
    with zipfile.ZipFile(ZIP_ATLAS, 'r') as zip_ref:
        zip_ref.extractall(RAW_DIR_ATLAS)
        os.remove(ZIP_ATLAS) 
    print(f'Dataset FractAtlas extraído en {RAW_DIR_ATLAS}')
else:
    print(f'Archivo {ZIP_ATLAS} no encontrado. Asegúrate de haberlo descargado en la celda con !curl.')


In [ ]:
tasks_fractured_at = collect_tasks(os.path.join(RAW_DIR_ATLAS, 'images/Fractured'))
tasks_non_fractured_at = collect_tasks(os.path.join(RAW_DIR_ATLAS, 'images/Non_fractured'))

results = []
for src_path, filename in tqdm(tasks_fractured_at, total=len(tasks_fractured_at), desc="Procesando imágenes fracturadas"):
    result = process_and_store_fracatlas(
        src_path, filename,
        target_dir=TARGET_DIR_FRACTURED,
        prefix="fa_f"
    )
    results.append(result)

for src_path, filename in tqdm(tasks_non_fractured_at, total=len(tasks_non_fractured_at), desc="Procesando imágenes no fracturadas"):
    result = process_and_store_fracatlas(
        src_path, filename,
        target_dir=TARGET_DIR_NON_FRACTURED,
        prefix="fa_nf"
    )
    results.append(result)

# Resumen
procesadas_exito = [r for r in results if r is True]
omitidas = [r for r in results if isinstance(r, str) and "Omitida" in r]
errores = [r for r in results if isinstance(r, str) and "Error" in r]

print(f'\nProcesadas con éxito: {len(procesadas_exito)}')
print(f'Omitidas por calidad: {len(omitidas)}')

if errores:
    print(f'Errores críticos: {len(errores)}')

## Descarga y preprocesamiento de dataset BoneFracture

In [ ]:
# Descargar el dataset desde Kaggle utilizando la API de Kaggle
!curl -L -o RAW_PATH/bonebreak.zip\
  https://www.kaggle.com/api/v1/datasets/download/pkdarabi/bone-break-classification-image-dataset

In [ ]:
# Configuración de rutas
ZIP_BONE = RAW_PATH / 'bonebreak.zip'
RAW_DIR_BONE = PROCESSED_PATH / 'BoneBreak_raw'

# 1. Extraer el nuevo dataset
os.makedirs(RAW_DIR_BONE, exist_ok=True)

if os.path.exists(ZIP_BONE):
    with zipfile.ZipFile(ZIP_BONE, 'r') as zip_ref:
        zip_ref.extractall(RAW_DIR_BONE)
    print(f'Dataset BoneBreak extraído en {RAW_DIR_BONE}')
else:
    print(f'Archivo {ZIP_BONE} no encontrado. Asegúrate de haberlo descargado en la celda con !curl.')


In [ ]:
tasks_fractured_bf = collect_tasks(os.path.join(RAW_DIR_BONE))

results = []
for src_path, filename in tqdm(tasks_fractured_bf, total=len(tasks_fractured_bf), desc="Procesando imágenes fracturadas"):
    result = process_and_store_fracatlas(
        src_path, filename,
        target_dir=TARGET_DIR_FRACTURED,
        prefix="fa_f"
    )
    results.append(result)

# Resumen
procesadas_exito = [r for r in results if r is True]
omitidas = [r for r in results if isinstance(r, str) and "Omitida" in r]
errores = [r for r in results if isinstance(r, str) and "Error" in r]

print(f'\nProcesadas con éxito: {len(procesadas_exito)}')
print(f'Omitidas por calidad: {len(omitidas)}')

if errores:
    print(f'Errores críticos: {len(errores)}')